# Train SFT, two DPO models, and persona LoRA

Attach the supplied project zip as a Kaggle dataset. Enable internet and **GPU T4 x2**. Run stages separately and save a notebook version after each completed stage. This notebook has not been run on Kaggle yet.

In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys, zipfile
REPO = Path('/kaggle/working/small-model-alignment')
if not REPO.exists():
    archives = list(Path('/kaggle/input').rglob('small-model-alignment.zip'))
    if len(archives) == 1:
        with zipfile.ZipFile(archives[0]) as archive:
            archive.extractall('/kaggle/working')
    else:
        sources = [p.parent for p in Path('/kaggle/input').rglob('submission_adapter.py')
                   if (p.parent / 'alignment/core.py').is_file()
                   and (p.parent / 'configs/sft.json').is_file()
                   and not (p.parent / 'outputs/sft/complete.json').exists()]
        if len(sources) != 1:
            raise RuntimeError('Attach exactly one project dataset containing the ZIP or unpacked project files')
        shutil.copytree(sources[0], REPO)
os.chdir(REPO)
sys.path.insert(0, str(REPO))
def run(script, *arguments):
    subprocess.run([sys.executable, script, *map(str, arguments)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', '-q', 'torchao'], check=False)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.', '--no-deps'], check=True)


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-persona.txt'], check=True)

In [ ]:
import torch, transformers, datasets
print('torch', torch.__version__, 'transformers', transformers.__version__, 'datasets', datasets.__version__)
assert torch.cuda.device_count() >= 2, 'Select GPU T4 x2'
for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i))
subprocess.run(['nvidia-smi'], check=True)
subprocess.run([sys.executable, '-m', 'pytest', '-q'], check=True)


## Prepare exact subsets

Defaults are 30,000 SFT examples, 15,000 pairs per manual DPO track, 1,000 validation rows per track, and 300 AlpacaEval prompts with seed 42. Preparation fails explicitly if filtering leaves too few rows.

In [ ]:
run('scripts/prepare_data.py')

## SFT

For an interrupted run, add `--resume` to the command. Completed stages should be skipped when rerunning; their `complete.json` files confirm saved best/final models.

In [ ]:
if not Path('outputs/sft/complete.json').exists():
    run('scripts/train.py', '--config', 'configs/sft.json', *(['--resume'] if Path('outputs/sft/resume/training_state.pt').exists() else []))
run('scripts/plot_metrics.py', '--run-dir', 'outputs/sft')

## Two independent manual DPO runs

Both begin from the same SFT best checkpoint. The policy runs on GPU 0 and the frozen reference on GPU 1.

In [ ]:
for track in ['hh', 'uf']:
    name = f'dpo_{track}'
    if not Path(f'outputs/{name}/complete.json').exists():
        run('scripts/train.py', '--config', f'configs/{name}.json', *(['--resume'] if Path(f'outputs/{name}/resume/training_state.pt').exists() else []))
    run('scripts/plot_metrics.py', '--run-dir', f'outputs/{name}')

## Persona LoRA

Run as a single process on GPU 0 so the configured effective batch size remains 8. This is the only stage that uses TRL. The install preserves the existing torch installation.

In [ ]:
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
if not Path('outputs/persona/complete.json').exists():
    checkpoints = sorted(Path('outputs/persona').glob('checkpoint-*'), key=lambda p: int(p.name.split('-')[-1]))
    run('scripts/train_persona.py', *(['--resume-from-checkpoint', checkpoints[-1]] if checkpoints else []))
run('scripts/plot_metrics.py', '--run-dir', 'outputs/persona')

## Preserve results

Use **Save Version** and preserve all notebook output files. Attach this saved training output to the generation notebook. Checkpoints, metric logs, configs, and prepared subsets live in `/kaggle/working/small-model-alignment`. Save the training version before proceeding. Avoid creating a second full archive of all checkpoints on the limited Kaggle disk.